# Challenge de la Sesión 04 — Afina, compara y decide

**Afinamiento de modelos generativos** · *Desarrollo y Evaluación de Modelos Propios* · Universidad del Rosario · Rosario GSB

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/04-fine-tuning/challenge.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/04-fine-tuning/challenge.ipynb)

| | |
|---|---|
| **Modalidad** | Individual |
| **Tiempo** | 40 minutos en clase · entrega hasta las **23:59 de hoy** (hora de Colombia) |
| **Uso de IA** | Permitido, con **registro obligatorio** al final del notebook ([política de uso de IA](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/politica-uso-ia.md)) |
| **Micro-sustentaciones** | Durante el challenge se eligen 3–4 estudiantes al azar para explicar su notebook en 2–3 minutos, sin IA. Si no puedes explicar tu entrega, la nota máxima de este challenge es **3.0** |
| **Rúbrica** | Ejecución técnica **40%** · Análisis e interpretación **40%** · Reflexión y registro de IA **20%** ([evaluación](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/evaluacion.md)) |
| **Entrega** | `Archivo → Descargar notebook` (*File → Download notebook*) y súbelo a **e-Aulas** como **`S04_<codigo>.ipynb`** (por ejemplo, `S04_123456.ipynb`) |

### Cómo trabajar
1. En Kaggle, activa **Settings → Accelerator → GPU T4 x2** y **Settings → Internet → On**. Con GPU, cada entrenamiento tarda menos de un minuto; en CPU funciona, pero tarda varios minutos.
2. Escribe tu **código estudiantil** y tu nombre en la primera celda de código y ejecuta el notebook **en orden** (*Run All*).
   Tu configuración personal (10 intents de Banking77, dos configuraciones de entrenamiento y un dominio de negocio) depende de tu código.
3. Completa las celdas marcadas con ✏️ y responde en las celdas que empiezan con **Tu respuesta:**, citando **tus** números.
4. Antes de entregar, verifica que aparezcan **"Tu configuración personal"** y la **huella de resultados** al final.

| Tarea | Qué haces | Componente de la rúbrica |
|---|---|---|
| 1 · Ejecución | Afinas DistilBERT en tus 10 intents con tu configuración A y reportas accuracy y F1 macro | Ejecución técnica (40%) |
| 2 · Análisis | Entrenas la configuración B, comparas curvas, diagnosticas si hay subajuste, sobreajuste o ninguno de los dos, eliges con validación y evalúas en test una sola vez | Análisis e interpretación (40%) |
| 3 · Explica y decide | Recomiendas afinamiento, RAG o *prompting* para tu dominio asignado, con tus números | Análisis e interpretación (40%) |
| 4 · Crítica a la IA | Encuentras, explicas y corriges 4 errores en unos `TrainingArguments` propuestos por un asistente | Análisis e interpretación (40%) |
| Reflexión y registro de IA | Al final del notebook | Reflexión y registro de IA (20%) |

In [ ]:
# ✏️ Escribe tu código estudiantil (como aparece en e-Aulas) y tu nombre completo.
STUDENT_ID = ""
STUDENT_NAME = ""

# --- Configuración del curso (no editar) ---
import os, sys, random, subprocess, importlib, hashlib, json
SESSION = "S04"
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules
if FAST_DEV_RUN and not STUDENT_ID.strip():
    STUDENT_ID, STUDENT_NAME = "000000", "Prueba automática"
assert STUDENT_ID.strip(), "⚠️ Escribe tu STUDENT_ID en la primera línea de esta celda y vuelve a ejecutarla."

def ensure(package, import_name=None):
    """Instala un paquete solo si falta."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

PERSONAL_SEED = int(hashlib.sha256(f"{SESSION}:{STUDENT_ID.strip()}".encode()).hexdigest()[:8], 16)
rng = random.Random(PERSONAL_SEED)

def pick(options):
    """Elige una opción de forma determinística a partir de tu código estudiantil."""
    return rng.choice(list(options))

import numpy as np
import torch
random.seed(PERSONAL_SEED); np.random.seed(PERSONAL_SEED % (2**32)); torch.manual_seed(PERSONAL_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sesión {SESSION} | Estudiante: {STUDENT_NAME} ({STUDENT_ID}) | Semilla personal: {PERSONAL_SEED} | Dispositivo: {DEVICE}")
# --- fin de la configuración ---

# Paquetes de esta sesión
ensure("transformers")
ensure("datasets")
ensure("accelerate")
ensure("scikit-learn", "sklearn")

In [ ]:
# --- Datos: Banking77 (no editar) ---
from datasets import load_dataset, ClassLabel

raw = load_dataset("legacy-datasets/banking77")
TODOS_LOS_INTENTS = raw["train"].features["label"].names
print(f"Banking77: {len(raw['train'])} ejemplos de entrenamiento, {len(raw['test'])} de test, {len(TODOS_LOS_INTENTS)} intents")

In [ ]:
# --- Tu configuración personal (no editar) ---
LR_OPCIONES = [2e-5, 5e-5, 1e-4]
EPOCAS_OPCIONES = [1, 2, 3]
REJILLA = [{"lr": lr, "epochs": ep} for lr in LR_OPCIONES for ep in EPOCAS_OPCIONES]

DOMINIOS = {  # Todas las organizaciones son ficticias.
    "Andes Bank · enrutamiento de mensajes": (
        "60.000 mensajes al mes desde la app deben llegar al equipo correcto (10 equipos). Hay 25.000 mensajes históricos "
        "ya clasificados por los agentes. Las categorías cambian poco. Se exige responder en menos de 200 ms y los mensajes "
        "contienen datos personales."),
    "NovaTel · preguntas sobre planes y tarifas": (
        "Un asistente responde 3.000 preguntas diarias sobre 120 planes móviles. El catálogo de planes y precios cambia cada mes. "
        "No hay conversaciones etiquetadas. Las respuestas deben citar la tarifa vigente."),
    "Banco Andino · voz de marca en respuestas escritas": (
        "Los agentes redactan 15.000 respuestas al mes y marketing aprobó 800 respuestas modelo con un tono muy particular. "
        "Las políticas de fondo casi no cambian. El banco quiere que un modelo proponga borradores con esa voz."),
    "Aseguradora Cóndor · clasificación de reclamaciones": (
        "12.000 reclamaciones ya etiquetadas en 40 categorías internas con jerga propia. Incluyen datos de salud y, por "
        "política interna, no pueden salir de la nube privada de la empresa. Volumen: 1.500 reclamaciones nuevas por semana."),
    "Café Alto Montaña · publicaciones para redes sociales": (
        "Una marca pequeña publica unas 20 piezas por semana en redes. No tiene equipo técnico ni datos históricos organizados. "
        "No maneja datos sensibles."),
    "Cooperativa Pradera · asistente de políticas de crédito": (
        "Los asesores consultan un manual de crédito de 400 páginas que se actualiza cada trimestre. Necesitan respuestas que "
        "citen la sección exacta del manual. Hay unas 200 preguntas frecuentes documentadas, sin respuestas etiquetadas."),
}

intent_ids = sorted(rng.sample(range(len(TODOS_LOS_INTENTS)), 10))
config_a = pick(REJILLA)
config_b = pick([c for c in REJILLA if c != config_a])
dominio = pick(sorted(DOMINIOS))

CONFIG = {
    "intents": [TODOS_LOS_INTENTS[i] for i in intent_ids],
    "config_a": config_a,
    "config_b": config_b,
    "domain": dominio,
}

print("🎯 Tu configuración personal")
print("   Intents:", ", ".join(CONFIG["intents"]))
print(f"   Configuración A: learning_rate={config_a['lr']:g}, épocas={config_a['epochs']}")
print(f"   Configuración B: learning_rate={config_b['lr']:g}, épocas={config_b['epochs']}")
print(f"   Dominio para la Tarea 3: {dominio}\n   → {DOMINIOS[dominio]}")

## Tarea 1 — Ejecución: afina DistilBERT con tu configuración A (Ejecución técnica, 40%)

Vas a repetir la Parte 2 del lab con **tus** 10 intents:

1. La celda de apoyo filtra Banking77 a tus intents, los renumera 0–9 y crea una **validación estratificada** (15% de `train`).
   El `test` oficial se usa **una sola vez**, en la Tarea 2. También imprime la distribución de longitudes en tokens.
2. En la celda ✏️ elige `MAX_LENGTH` a partir del percentil 95 impreso (el valor por defecto funciona).
3. Ejecuta el entrenamiento con tu **configuración A**. Todo lo demás queda fijo (misma semilla, mismo modelo, mismos datos),
   así que en la Tarea 2 la única diferencia entre A y B será la tasa de aprendizaje y el número de épocas.

In [ ]:
# --- Celda de apoyo: datos de tus intents (no editar) ---
import math, time
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding, set_seed)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

INTENTS = CONFIG["intents"]
VIEJO_A_NUEVO = {TODOS_LOS_INTENTS.index(n): i for i, n in enumerate(INTENTS)}

def filtrar(split):
    ds = raw[split].filter(lambda ex: ex["label"] in VIEJO_A_NUEVO)
    features = ds.features.copy()
    features["label"] = ClassLabel(names=INTENTS)
    return ds.map(lambda ex: {"label": VIEJO_A_NUEVO[ex["label"]]}, features=features)

def por_clase(ds, n, seed=42):
    """Submuestra estratificada: como máximo n ejemplos por intent (todos si n es None)."""
    if n is None:
        return ds
    filas = {}
    for i, y in enumerate(list(ds["label"])):
        filas.setdefault(y, []).append(i)
    r = random.Random(seed)
    elegidas = []
    for y in sorted(filas):
        r.shuffle(filas[y])
        elegidas += filas[y][:n]
    return ds.select(sorted(elegidas))

particion = filtrar("train").train_test_split(test_size=0.15, stratify_by_column="label", seed=42)
train_ds = por_clase(particion["train"], 15 if FAST_DEV_RUN else None)
val_ds = por_clase(particion["test"], 5 if FAST_DEV_RUN else None)
test_ds = por_clase(filtrar("test"), 10 if FAST_DEV_RUN else None)
print(f"Entrenamiento: {len(train_ds)} · validación: {len(val_ds)} · test: {len(test_ds)}")
print(f"Con {len(val_ds)} ejemplos de validación, un error más o menos mueve la accuracy {100 / len(val_ds):.2f} puntos.")

MODEL_NAME = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
longitudes = np.array([len(ids) for ids in tokenizer(list(train_ds["text"]))["input_ids"]])
print(f"Tokens por mensaje: mediana {np.median(longitudes):.0f} · percentil 95 {np.percentile(longitudes, 95):.0f} · máximo {longitudes.max()}")

In [ ]:
# ✏️ Tarea 1: elige MAX_LENGTH a partir del percentil 95 de arriba (debe cubrir ≈95% de los mensajes).
MAX_LENGTH = 32 if FAST_DEV_RUN else 64
BATCH_SIZE = 16 if FAST_DEV_RUN else 32

def tokenizar(lote):
    return tokenizer(lote["text"], truncation=True, max_length=MAX_LENGTH)

train_tok = train_ds.map(tokenizar, batched=True)
val_tok = val_ds.map(tokenizar, batched=True)
test_tok = test_ds.map(tokenizar, batched=True)
print(f"MAX_LENGTH = {MAX_LENGTH}: se truncan {np.mean(longitudes > MAX_LENGTH):.1%} de los mensajes de entrenamiento")

In [ ]:
# --- Celda de apoyo: función de entrenamiento (no editar) ---
def metricas(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred), "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0)}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return metricas(labels, np.argmax(logits, axis=-1))

def entrenar(config, nombre):
    """Afina DistilBERT desde cero con la configuración dada y devuelve curvas, métricas de validación y tiempo."""
    set_seed(42)  # misma inicialización para todas las configuraciones
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(INTENTS), id2label=dict(enumerate(INTENTS)),
        label2id={n: i for i, n in enumerate(INTENTS)})
    epocas = min(config["epochs"], 1) if FAST_DEV_RUN else config["epochs"]
    pasos = math.ceil(len(train_tok) / BATCH_SIZE) * epocas
    args = TrainingArguments(
        output_dir=f"outputs/s04-config-{nombre}",
        learning_rate=config["lr"],
        num_train_epochs=epocas,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=64,
        warmup_steps=int(0.1 * pasos),
        weight_decay=0.01,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="no",
        report_to="none",
        seed=42,
        use_cpu=(DEVICE == "cpu"),
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok,
                      data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
    inicio = time.time()
    trainer.train()
    segundos = time.time() - inicio
    hist = pd.DataFrame(trainer.state.log_history)
    entrenamiento = hist.dropna(subset=["loss"])[["epoch", "loss"]].rename(columns={"loss": "train_loss"})
    validacion = hist.dropna(subset=["eval_loss"])[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]]
    curvas = validacion.merge(entrenamiento, on="epoch", how="left").rename(columns={
        "eval_loss": "val_loss", "eval_accuracy": "val_accuracy", "eval_f1_macro": "val_f1_macro"})
    ultima = curvas.iloc[-1]
    return {"nombre": nombre, "config": config, "epocas": epocas, "trainer": trainer, "curvas": curvas,
            "val": {"accuracy": float(ultima["val_accuracy"]), "f1_macro": float(ultima["val_f1_macro"])},
            "train_loss_final": float(ultima["train_loss"]), "val_loss_final": float(ultima["val_loss"]),
            "segundos": segundos}

def resumen(res):
    print(f"Configuración {res['nombre']}: lr={res['config']['lr']:g}, épocas={res['epocas']} · {res['segundos']:.0f} s en {DEVICE}")
    display(res["curvas"].round(4))
    pred = res["trainer"].predict(val_tok)
    y_true, y_pred = pred.label_ids, pred.predictions.argmax(-1)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(INTENTS))))
    pares = sorted([(cm[i, j], INTENTS[i], INTENTS[j]) for i in range(len(INTENTS))
                    for j in range(len(INTENTS)) if i != j and cm[i, j] > 0], reverse=True)[:3]
    print("Confusiones más frecuentes en validación (real → predicho):",
          "; ".join(f"{a} → {b} ({c})" for c, a, b in pares) or "ninguna")

In [ ]:
# ✏️ Tarea 1: entrena tu configuración A (no cambies CONFIG; si quieres experimentar, hazlo en una celda nueva).
res_a = entrenar(CONFIG["config_a"], "A")
resumen(res_a)

**Tu respuesta:**

1. ¿Qué `MAX_LENGTH` usaste y por qué? Cita el percentil 95 y el porcentaje de mensajes truncados.
2. Accuracy y F1 macro de **validación** de tu configuración A (cópialos de la tabla).
3. ¿Qué par de intents se confunde más? Lee sus nombres: ¿por qué crees que el modelo los confunde?

## Tarea 2 — Análisis: configuración B, curvas y una sola evaluación en test (Análisis e interpretación, 40%)

1. Entrena tu **configuración B** con la misma función.
2. Compara las curvas de pérdida de entrenamiento y validación por época de A y B, y diagnostica **cada** una con lo que muestran: ¿subajuste, buen ajuste, sobreajuste o divergencia? No des por hecho que alguno de estos problemas aparece: justifícalo con tus curvas.
3. La celda elige la configuración con **mejor F1 macro de validación** y solo esa se evalúa en **test**, una vez.
   Si decides elegir la otra (por ejemplo, porque la diferencia es menor que un error de validación y B es más barata), cambia `ELEGIDA` y justifícalo.

In [ ]:
# ✏️ Tarea 2: entrena la configuración B y compara.
res_b = entrenar(CONFIG["config_b"], "B")
resumen(res_b)

comparacion = pd.DataFrame([
    {"config": r["nombre"], "learning_rate": r["config"]["lr"], "épocas": r["epocas"],
     "val_accuracy": r["val"]["accuracy"], "val_f1_macro": r["val"]["f1_macro"],
     "train_loss_final": r["train_loss_final"], "val_loss_final": r["val_loss_final"], "segundos": r["segundos"]}
    for r in (res_a, res_b)]).set_index("config").round(4)
display(comparacion)

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for eje, r in zip(ejes, (res_a, res_b)):
    c = r["curvas"]
    eje.plot(c["epoch"], c["train_loss"], "o-", label="pérdida de entrenamiento")
    eje.plot(c["epoch"], c["val_loss"], "s--", label="pérdida de validación")
    eje.set_title(f"Config {r['nombre']}: lr={r['config']['lr']:g}, {r['epocas']} época(s)")
    eje.set_xlabel("época"); eje.legend()
ejes[0].set_ylabel("pérdida")
plt.tight_layout(); plt.show()

ELEGIDA = "A" if res_a["val"]["f1_macro"] >= res_b["val"]["f1_macro"] else "B"   # ✏️ criterio: F1 macro de validación
res_elegida = res_a if ELEGIDA == "A" else res_b
pred_test = res_elegida["trainer"].predict(test_tok)
TEST = metricas(pred_test.label_ids, pred_test.predictions.argmax(-1))
print(f"Configuración elegida con validación: {ELEGIDA} → test (una sola vez): "
      f"accuracy {TEST['accuracy']:.4f} · F1 macro {TEST['f1_macro']:.4f}")

**Tu respuesta:**

1. Diagnóstico de **cada** configuración (subajuste, buen ajuste, sobreajuste o divergencia), con evidencia de las curvas y de la tabla.
2. ¿La diferencia de F1 macro entre A y B es mayor que el efecto de un solo error de validación (impreso en la Tarea 1)? ¿Es una diferencia confiable?
3. ¿Qué configuración elegiste y por qué? Compara su resultado de test con el de validación: ¿se parecen? ¿Qué significaría que test fuera mucho peor?
4. Si tuvieras que entrenar una tercera configuración, ¿cuál probarías y qué esperas que pase?

## Tarea 3 — Explica y decide: ¿afinamiento, RAG o *prompting*? (Análisis e interpretación, 40%)

Tu dominio asignado aparece abajo. Recomienda **una** estrategia (o una combinación justificada) usando la guía de decisión de las slides:
*prompting* como línea base, RAG para conocimiento que cambia, afinamiento para comportamiento (formato, tono, clasificación estrecha, costo a escala).
Usa **tus** números como evidencia de lo que cuesta y logra un afinamiento: cuántos ejemplos etiquetados usaste, cuánto tardó y qué F1 obtuviste.

In [ ]:
# --- Tus números para la Tarea 3 (no editar) ---
print(f"Dominio: {CONFIG['domain']}\n→ {DOMINIOS[CONFIG['domain']]}\n")
print(f"Ejemplos etiquetados usados para entrenar: {len(train_ds)} ({len(train_ds) // len(INTENTS)} por intent en promedio)")
print(f"Tiempo de entrenamiento de la configuración elegida ({ELEGIDA}): {res_elegida['segundos']:.0f} s en {DEVICE}")
print(f"F1 macro en test de tu modelo afinado: {TEST['f1_macro']:.4f}")

**Tu respuesta:**

- **Decisión** (*prompting*, RAG, afinamiento completo, LoRA o combinación):
- **Por qué**, con al menos **dos** números (tuyos o del dominio):
- **Qué datos necesitarías** y cuánto costaría conseguirlos o etiquetarlos (estimación simple):
- **Principal riesgo** (privacidad, conocimiento desactualizado, olvido, costo, dependencia del proveedor…) y **cómo lo medirías**:

## Tarea 4 — Crítica a la IA: unos `TrainingArguments` "óptimos" (Análisis e interpretación, 40%)

Un colega de Andes Bank tiene **200 mensajes etiquetados** en 10 intents y le pidió a un asistente de IA: *"Dame la mejor configuración
para afinar DistilBERT con Trainer"*. Esta fue la respuesta:

> Para un dataset pequeño como el tuyo, esta es la configuración óptima. Usa una tasa de aprendizaje alta para que el modelo
> converja más rápido, y muchas épocas para compensar que tienes pocos datos. No pierdas tiempo evaluando durante el entrenamiento;
> al final, evalúa cada época en el conjunto de test y quédate con la que tenga mejor F1: así te aseguras de entregar el mejor modelo.
>
> ```python
> args = TrainingArguments(
>     output_dir="outputs/andes-intents",
>     learning_rate=5e-2,             # alta = converge más rápido
>     num_train_epochs=50,            # más épocas compensan los pocos datos
>     per_device_train_batch_size=16,
>     weight_decay=0.01,
>     eval_strategy="no",             # evaluar solo hace lento el entrenamiento
>     save_strategy="epoch",
>     report_to="none",
> )
> trainer = Trainer(model=model, args=args, train_dataset=train_tok,
>                   data_collator=DataCollatorWithPadding(tokenizer))
> trainer.train()
> # Después: evalúa cada checkpoint en test_tok y quédate con el de mayor F1 macro.
> ```

La propuesta tiene **4 errores** de criterio. Para cada uno: cita la línea o frase, explica por qué es un error y propón la corrección.
Luego corrige la configuración en la celda ✏️ y verifica con los chequeos automáticos.

**Evidencia:** la primera celda entrena **1 época** con `learning_rate=5e-2` sobre tus datos, para que veas qué pasa (en GPU tarda segundos).

In [ ]:
# --- Evidencia: una época con la tasa de aprendizaje de la IA (no editar) ---
res_ia = entrenar({"lr": 5e-2, "epochs": 1}, "IA")
clase_mayoritaria = np.bincount(list(val_ds["label"])).max() / len(val_ds)   # accuracy de predecir siempre la clase más frecuente
print(f"lr=5e-2 → accuracy de validación {res_ia['val']['accuracy']:.4f} · F1 macro {res_ia['val']['f1_macro']:.4f} · "
      f"pérdida de validación {res_ia['val_loss_final']:.3f}")
print(f"Referencias: ln(10) ≈ {math.log(10):.3f} (pérdida de adivinar al azar entre 10 intents) · "
      f"predecir siempre la clase más frecuente de validación daría una accuracy de {clase_mayoritaria:.4f}")

In [ ]:
# ✏️ Tarea 4: corrige la configuración de la IA (hoy tiene los valores que ella propuso) y ejecuta la celda.
args_corregidos = TrainingArguments(
    output_dir="outputs/andes-intents",
    learning_rate=5e-2,
    num_train_epochs=50,
    per_device_train_batch_size=16,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    use_cpu=(DEVICE == "cpu"),
)
CONJUNTO_PARA_ELEGIR = "test"   # ✏️ ¿con qué conjunto se elige la mejor época o configuración? ("test" o "validation")

# --- Chequeos automáticos (no editar) ---
estrategia = getattr(args_corregidos.eval_strategy, "value", args_corregidos.eval_strategy)
chequeos = {
    "learning_rate: razonable para afinamiento completo": 1e-5 <= args_corregidos.learning_rate <= 1e-4,
    "num_train_epochs: razonable para ~200 ejemplos": 2 <= args_corregidos.num_train_epochs <= 5,
    "se evalúa durante el entrenamiento (eval_strategy distinto de 'no')": estrategia != "no",
    "la selección se hace con validación, no con test": CONJUNTO_PARA_ELEGIR == "validation",
}
for texto, ok in chequeos.items():
    print(("✅ " if ok else "❌ ") + texto)
T4_CHEQUEOS_OK = sum(chequeos.values())
print(f"Chequeos superados: {T4_CHEQUEOS_OK}/4")

**Tu respuesta:**

| # | Línea o frase de la IA | Por qué es un error | Corrección | Evidencia (número, curva o fuente) |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |
| 4 |  |  |  |  |

**¿Qué dato de tu propio notebook usarías para convencer a tu colega?**

In [ ]:
# --- Resultados clave (no editar: se llenan con tus variables) ---
def r4(x):
    return round(float(x), 4)

def resultado_config(res):
    return {"lr": res["config"]["lr"], "epocas_ejecutadas": res["epocas"],
            "val_accuracy": r4(res["val"]["accuracy"]), "val_f1_macro": r4(res["val"]["f1_macro"]),
            "train_loss_final": r4(res["train_loss_final"]), "val_loss_final": r4(res["val_loss_final"])}

RESULTS = {
    "max_length": MAX_LENGTH,
    "n_train": len(train_ds), "n_val": len(val_ds), "n_test": len(test_ds),
    "config_a": resultado_config(res_a),
    "config_b": resultado_config(res_b),
    "config_elegida": ELEGIDA,
    "test_accuracy": r4(TEST["accuracy"]),
    "test_f1_macro": r4(TEST["f1_macro"]),
    "t4_lr_alta_val_accuracy": r4(res_ia["val"]["accuracy"]),
    "t4_chequeos_ok": int(T4_CHEQUEOS_OK),
}
print("✅ RESULTS listo. Ejecuta la última celda para ver tu huella de resultados.")

## 🪞 Reflexión y registro de uso de IA (20%)

### Reflexión (responde en 3–5 líneas cada una)
1. ¿Qué aprendiste hoy que puedas aplicar en tu trabajo u organización?
2. ¿Qué fue lo más difícil y cómo lo resolviste?
3. Autoevaluación: ¿qué nota (0.0–5.0) te pondrías en este challenge y por qué?

### 🤖 Registro de uso de IA (obligatorio)
| # | Herramienta (ChatGPT, Claude, Codex, Copilot…) | ¿Para qué la usaste? | Prompt principal (resumido) | ¿Qué verificaste o corregiste de su respuesta? |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |

> Si no usaste asistentes de IA, escribe: *"No usé asistentes de IA en este challenge."* Recuerda: puedes usar IA, pero **eres responsable de todo lo que entregas** y podrías ser seleccionado(a) para una micro-sustentación.

In [ ]:
# --- Huella de resultados (no editar) ---
def results_fingerprint(results):
    payload = json.dumps({"session": SESSION, "student": STUDENT_ID.strip(), "results": results},
                         sort_keys=True, default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()[:12]

print(json.dumps({"CONFIG": CONFIG, "RESULTS": RESULTS}, indent=2, ensure_ascii=False, default=str))
print("🔏 Huella de resultados:", results_fingerprint({"CONFIG": CONFIG, "RESULTS": RESULTS}))